In [1]:
import warnings

warnings.filterwarnings("ignore")
import shutil
import parmed as pmd
from pathlib import Path
import foyer
import json
import mbuild as mb
import gmso
from gmso.external import from_parmed

from parmed.exceptions import MoleculeError, FormatNotFound
from foyer.exceptions import MissingParametersError

import sTree

ModuleNotFoundError: No module named 'sTree'

# Parsing OPLS

In [2]:
foyer_opls = foyer.forcefield.Forcefield(name="oplsaa")
opls_dir = Path("./files")

opls_atom_types = dict()
opls_bond_types = dict()
opls_angle_types = dict()
opls_harmonic_dih_types = dict()
opls_rb_dih_types = dict()
opls_improper_types = dict()
smart_strings = dict()
error_files = []
harm_and_rb_dihedral_files = []
harm_only_files = []
actual_new_atom_types = []

In [3]:
for top_file in opls_dir.glob("*.top"):
    found_harmonic_dih = False
    found_rb_dih = False
    try:
        struc = pmd.load_file(filename=str(top_file))
        # Parse atom types
        for atom in struc.atoms:
            opls_atom_types[atom.atom_type.name] = dict(
                sigma=atom.atom_type.sigma / 10,
                epsilon=atom.atom_type.epsilon,
                charge=atom.atom_type.charge,
                file=str(top_file),
                path=top_file
            )
        
        # Parse harmonic bonds
        for bond in struc.bonds:
            bond_type = f"{bond.atom1.atom_type}-{bond.atom2.atom_type}"
            opls_bond_types[bond_type] = dict(req=bond.type.req / 10, k=bond.type.k)
        
        # Parse harmonic angles
        for angle in struc.angles:
            angle_type = f"{angle.atom1.atom_type}-{angle.atom2.atom_type}-{angle.atom3.atom_type}"
            opls_angle_types[angle_type] = dict(
                thetaeq=angle.type.theteq, k=angle.type.k
            )

        # Parse harmonic dihedrals
        for dih in struc.dihedrals:
            dihedral_type = f"{dih.atom1.atom_type}-{dih.atom2.atom_type}-{dih.atom3.atom_type}-{dih.atom4.atom_type}"
            opls_harmonic_dih_types[dihedral_type] = dict(
                k=dih.type.phi_k, phase=dih.type.phase
            )
            found_harmonic_dih = True

        # Parse RB dihedrals
        for rb in struc.rb_torsions:
            rb_type = f"{rb.atom1.atom_type}-{rb.atom2.atom_type}-{rb.atom3.atom_type}-{rb.atom4.atom_type}"
            opls_rb_dih_types[rb_type] = dict(
                c0=rb.type.c0,
                c1=rb.type.c1,
                c2=rb.type.c2,
                c3=rb.type.c3,
                c4=rb.type.c4,
                c5=rb.type.c5,
            )
            found_rb_dih = True
    except (MoleculeError, ValueError, FormatNotFound) as e:
        print(f"Error: {e}")
        error_files.append(str(top_file))

    if all([found_harmonic_dih, found_rb_dih]):
        harm_and_rb_dihedral_files.append(str(top_file))
    elif found_harmonic_dih is True and found_rb_dih is False:
        harm_only_files.append(str(top_file))

Error: Cannot exclude an atom from itself! Atoms are: <Atom N [5]; In MOL 5> <Atom N [5]; In MOL 5>
Error: Cannot exclude an atom from itself! Atoms are: <Atom N [12]; In MOL 0> <Atom N [12]; In MOL 0>
Error: Cannot exclude an atom from itself! Atoms are: <Atom N [6]; In MOL 6> <Atom N [6]; In MOL 6>
Error: invalid literal for int() with base 10: 'formamide'
Error: Cannot exclude an atom from itself! Atoms are: <Atom N [8]; In MOL 0> <Atom N [8]; In MOL 0>
Error: Cannot exclude an atom from itself! Atoms are: <Atom N [14]; In MOL 14> <Atom N [14]; In MOL 14>


In [4]:
# Look through ffnonbonded.itp to get classes for the new atom types in the first dict
# Try reverse for angles, bonds and dihedrals

new_atom_types = dict()
new_bond_types = dict()
new_angle_types = dict()
new_harmonic_dih_types = dict()
new_rb_dih_types = dict()


for atom_type in opls_atom_types:
    if atom_type not in foyer_opls.atomTypeDefinitions.keys():
        new_atom_types[atom_type] = opls_atom_types[atom_type]
    if atom_type not in foyer_opls.atomTypeClasses:
        actual_new_atom_types.append(atom_type)

for bond_type in opls_bond_types:
    atom1 = bond_type.split("-")[0]
    atom2 = bond_type.split("-")[1]
    class1 = foyer_opls.atomTypeClasses[atom1]
    class2 = foyer_opls.atomTypeClasses[atom2]
    found = False
    for _bond in [[atom1, atom2], [atom2, atom1], [class1, class2], [class2, class1]]:
        try:
            bond_params = foyer_opls.get_parameters(
                group="harmonic_bonds", key=[_bond[0], _bond[1]]
            )
            found = True
        except MissingParametersError:
            pass
    if not found:
        new_bond_types[bond_type] = opls_bond_types[bond_type]

for angle_type in opls_angle_types:
    atom1 = angle_type.split("-")[0]
    atom2 = angle_type.split("-")[1]
    atom3 = angle_type.split("-")[2]
    class1 = foyer_opls.atomTypeClasses[atom1]
    class2 = foyer_opls.atomTypeClasses[atom2]
    class3 = foyer_opls.atomTypeClasses[atom3]
    found = False
    for _angle in [[atom1, atom2, atom3], [atom3, atom2, atom1], [class1, class2, class3], [class3, class2, class1]]:
        try:
            angle_params = foyer_opls.get_parameters(
                group="harmonic_angles", key=[_angle[0], _angle[1], _angle[2]]
            )
            found = True
        except MissingParametersError:
            pass
    if not found:
        new_angle_types[angle_type] = opls_angle_types[angle_type]

for rb_type in opls_rb_dih_types:
    atom1 = rb_type.split("-")[0]
    atom2 = rb_type.split("-")[1]
    atom3 = rb_type.split("-")[2]
    atom4 = rb_type.split("-")[3]
    class1 = foyer_opls.atomTypeClasses[atom1]
    class2 = foyer_opls.atomTypeClasses[atom2]
    class3 = foyer_opls.atomTypeClasses[atom3]
    class4 = foyer_opls.atomTypeClasses[atom4]
    found = False
    for _rb in [
        [atom1, atom2, atom3, atom4],
        [atom4, atom3, atom2, atom1],
        [class1, class2, class3, class4],
        [class4, class3, class2, class1]
    ]:
        try:
            params = foyer_opls.get_parameters(
                group="rb_propers", key=[_rb[0], _rb[1], _rb[2], _rb[3]]
            )
            found = True
        except MissingParametersError:
            pass
    if not found:
        new_rb_dih_types[rb_type] = opls_rb_dih_types[rb_type]

In [5]:
print(f"New Atom Types: {len(new_atom_types)}")
print(f"New Bond Types: {len(new_bond_types)}")
print(f"New Angle Types: {len(new_angle_types)}")
print(f"New RB Tors Types: {len(new_rb_dih_types)}")

New Atom Types: 73
New Bond Types: 153
New Angle Types: 329
New RB Tors Types: 444


In [6]:
def get_matching_pdb(top_file):
    base_name = top_file.stem
    matching_pdbs = list(opls_dir.glob(f"{base_name}*-gas.pdb"))
    return [str(f) for f in matching_pdbs]

In [28]:
def visualize_atom_type(atom_type, index=None):
    print(atom_type["path"])
    pdbs = get_matching_pdb(atom_type["path"])
    #struc = pmd.load_file(pdbs[0], )
    comp = mb.load(pdbs[0])
    print(pdbs)
    if index:
        for i, p in enumerate(comp.children[0].particles()):
            if i == index - 1:
                p.name = "_X"
    return comp.children[0]

In [19]:
# Test loading files

In [40]:
comp = mb.load("files/100-66-3-gas.pdb")
top = pmd.load_file("files/100-66-3.top")

def get_mol2(atom_type):
    atom_type_dict = new_atom_types[atom_type]
    pdbs = get_matching_pdb(atom_type_dict["path"])
    compound = mb.load(pdbs[0])
    struc = compound.to_parmed()
    top_struc = pmd.load_file(str(atom_type_dict["path"]))
    top_struc.coordinates = struc.coordinates
    compound = mb.load(top_struc)
    name = [key for key in top_struc.molecules.keys()][0]
    print(name)
    compound.save(f"testing_files/{name}.mol2", overwrite=True)
    shutil.copy(str(atom_type_dict["path"]), f"testing_files/{name}.top")
    new_atom_types[atom_type]["mol2_file"] = f"{name}.mol2"
    return compound, top_struc

In [16]:
for new_type in new_atom_types:
    comp, top = get_mol2(new_type)

acetophenone
bromomethane
cyclohexylamine
triethylamine
triethylamine
triethyl-phosphate
triethyl-phosphate
triethyl-phosphate
triethyl-phosphate
triethyl-phosphate
2-nitropropane
sulfolane
sulfolane
sulfolane
sulfolane
benzaldehyde
anisole
anisole
dimethoxymethane
dimethoxymethane
123-propanetriol
123-propanetriol
123-propanetriol
123-propanetriol
123-propanetriol
furan
furan
furan
furan
tert-butylamine
1-chloronaphthalene
2-iodopropane
2-methylpyridine
4-methylpyridine
diphenyl-ether
benzyl-alcohol
benzyl-alcohol
diphenyl-ether
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
quinoline
pyrrolidine
methyl-salicylate
1234-tetrafluorobenzene
1234-tetrafluorobenzene
12-difluorobenzene
12-difluorobenzene
benzenethiol
benzenethiol
2-chloroaniline
ethyl-vinyl-ether
3-methylpyridine
11-dichloroethene
11-dichloroethene
furan
trifluoromethyl-benzene
trifluoromethyl-benzene
trifluorome

In [27]:
print(get_matching_pdb(top_file=new_atom_types["opls_179"]["path"]))
print(new_atom_types["opls_179"]["mol2_file"])

['files/100-66-3-gas.pdb']
anisole.mol2


In [36]:
struc = visualize_atom_type(new_atom_types["opls_171"], index=1)
for p in struc.particles():
    print(p.element)
struc.visualize()

files/56-81-5.top
['files/56-81-5-gas.pdb']
Element: oxygen, symbol: O, atomic number: 8, mass: 15.999
Element: hydrogen, symbol: H, atomic number: 1, mass: 1.008
Element: carbon, symbol: C, atomic number: 6, mass: 12.011
Element: hydrogen, symbol: H, atomic number: 1, mass: 1.008
Element: hydrogen, symbol: H, atomic number: 1, mass: 1.008
Element: carbon, symbol: C, atomic number: 6, mass: 12.011
Element: hydrogen, symbol: H, atomic number: 1, mass: 1.008
Element: oxygen, symbol: O, atomic number: 8, mass: 15.999
Element: hydrogen, symbol: H, atomic number: 1, mass: 1.008
Element: carbon, symbol: C, atomic number: 6, mass: 12.011
Element: hydrogen, symbol: H, atomic number: 1, mass: 1.008
Element: hydrogen, symbol: H, atomic number: 1, mass: 1.008
Element: oxygen, symbol: O, atomic number: 8, mass: 15.999
Element: hydrogen, symbol: H, atomic number: 1, mass: 1.008


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
new_atom_types.keys()

dict_keys(['opls_233', 'opls_722', 'opls_912', 'opls_908', 'opls_902', 'opls_443', 'opls_444', 'opls_442', 'opls_440', 'opls_441', 'opls_765', 'opls_484', 'opls_485', 'opls_493', 'opls_494', 'opls_232', 'opls_199', 'opls_179', 'opls_189', 'opls_190', 'opls_171', 'opls_172', 'opls_173', 'opls_176', 'opls_174', 'opls_567', 'opls_569', 'opls_568', 'opls_570', 'opls_913', 'opls_147', 'opls_732', 'opls_670', 'opls_674', 'opls_473', 'opls_221', 'opls_218', 'opls_472', 'opls_610', 'opls_609', 'opls_618', 'opls_608', 'opls_617', 'opls_607', 'opls_616', 'opls_606', 'opls_615', 'opls_612', 'opls_605', 'opls_614', 'opls_604', 'opls_613', 'opls_603', 'opls_611', 'opls_619', 'opls_907', 'opls_471', 'opls_721', 'opls_720', 'opls_727', 'opls_728', 'opls_735', 'opls_734', 'opls_916', 'opls_518', 'opls_672', 'opls_227', 'opls_226', 'opls_566', 'opls_724', 'opls_725', 'opls_726', 'opls_914'])

In [38]:
new_opls = foyer.Forcefield(forcefield_files="oplsaa.xml")

In [21]:
new_atom_types["opls_199"]

{'sigma': 0.355,
 'epsilon': 0.06999999999999999,
 'charge': 0.085,
 'file': 'files/100-66-3.top',
 'path': PosixPath('files/100-66-3.top'),
 'mol2_file': 'anisole.mol2'}

In [20]:
new_opls.atomTypeDefinitions["opls_232"]

'[C;X3]([O;X1])([C;X3])[H]'

In [20]:
opls_atom_types["opls_722"]

{'sigma': 0.34700000000000003,
 'epsilon': 0.47,
 'charge': -0.22,
 'file': 'files/74-83-9.top',
 'path': PosixPath('files/74-83-9.top')}

In [53]:
for bond in struc.bonds():
    print(bond[0])

<O pos=([4.8506 4.9731 5.1038]), 1 bonds, id: 130879523696016>


## Notes

**opls_443**: Carbon from dimethyl phostate. The top file is for triethyl phosphate. How strict do we want to be with the SMARTS def?
We could do: Carbon bonded to an oxygen that is bonded to a phosphorous (with 4 oxygen bonds) How strict to be about other carbon bonds. 3 Hs (dimethyl phospate) or another methyl group (2 Hs and 1 C). I added it to work for either `[C;X4][O;X2][P;X4]([O;X2])([O;X2])(=O)`

**opls_232 and opls_233** These are very similar, double check SMARTS defs

# SMARTS DEFS

In [ ]:
files_dict = {}

for top_file in opls_dir.glob("*.top"):
    if str(top_file) in error_files:
        continue
    base_name = top_file.stem
    files_dict[str(top_file)] = []
    # Find matching pdb files
    matching_pdbs = list(opls_dir.glob(f"{base_name}-*.pdb"))
    for pdb_file in matching_pdbs:
        files_dict[str(top_file)].append(str(pdb_file))

In [ ]:
compounds = []
top_files = []
pdb_files = []

for top_file in files_dict:
    if top_file in error_files:
        continue
    try:
        pdb_file = files_dict[top_file][0]
        comp = mb.load(pdb_file)
        child = comp.children[0]
        compounds.append(child)
        top_files.append(top_file)
        pdb_files.append(pdb_file)
    except:
        print(top_file)

In [ ]:
for comp, top_file in zip(compounds, top_files):
    struc = comp.to_parmed()
    top = pmd.load_file(top_file)
    top.coordinates = struc.coordinates
    gmso_top = from_parmed(top)
    break
    

In [ ]:
comp = mb.load("CCC", smiles=True)

In [ ]:
for p in comp.particles_by_name("C"):
    p.name = "_X"
    p.element = None

hs = [p for p in comp.particles_by_name("H")]
for h in hs:
    comp.remove(h)

In [ ]:
BG = comp.bond_graph # the bond graph of our molecule, atoms connected by bonds represented as a Set of source atom to destination atoms

depth = 1 # this parameter determines how large the smart tree should be generated, the larger the depth the more specific your SMARTS definition is, but the more expensive it is to atomtype 
smarts_dict = sTree.bond_graph_to_smarts_dic(BG, depth) # returns our smarts in a dictionary

In [ ]:
for i in smarts_dict.values():
    if i in foyer_opls.atomTypeDefinitions:
        print(i, "Already in Foyer")
    else:
        print(i, "Not in foyer")